In [ ]:
#THIS CELL COMPUTES THE ERROR OF THE FIRST INTEGRAL WHEN USING THE COMPOSITE TRAPEZOID RULE

import numpy as np
import matplotlib.pyplot as plt

# 1. Define the integrand function and its integration limits
def f(x):
    return x / np.sqrt(x**2 + 9)

a, b = 0, 4
exact_value = 2.0  # Analytical solution: [sqrt(x^2+9)] from 0 to 4 = 5 - 3 = 2

# 2. Composite Trapezoid Rule function
def composite_trapezoid(f, a, b, m):
    x = np.linspace(a, b, m + 1)
    y = f(x)
    h = (b - a) / m
    integral = (h / 2) * (y[0] + 2 * np.sum(y[1:-1]) + y[-1])
    return integral

# 3. Calculate errors for step sizes h = (b-a)/2^k for k = 0 to 8
k_values = np.arange(0, 9)
m_values = 2**k_values
h_values = (b - a) / m_values
errors = []

for m in m_values:
    approx = composite_trapezoid(f, a, b, m)
    error = abs(exact_value - approx)
    errors.append(error)

errors = np.array(errors)

# 4. Calculate the empirical slope using linear regression on log-transformed data
# Exclude the first point if h is too large to represent asymptotic behavior accurately
slopes = np.polyfit(np.log(h_values[1:]), np.log(errors[1:]), 1)
empirical_slope = slopes[0]

print(f"Empirical slope from log-log data: {empirical_slope:.4f}")

# 5. Plot the log-log graph
plt.figure(figsize=(8, 5))
plt.loglog(h_values, errors, '-o', label='Trapezoid Rule Error', color='blue')
plt.loglog(h_values, h_values**2 * (errors[0]/h_values[0]**2), '--', label='Theoretical Slope O(h²)', color='red')

plt.xlabel('Step size ($h$)')
plt.ylabel('Absolute Error')
plt.title('Log-Log Plot of Error vs Step Size (Integral 1a)')
plt.grid(True, which="both", ls="--")
plt.legend()
plt.show()


In [ ]:
#THIS CELL COMPUTES THE ERRORS OF THE INTEGRALS USING THE COMPOSITE TRAPEZOID RULE

import numpy as np
import matplotlib.pyplot as plt

# 1. Define the integrands (a to h) and their analytical exact values
integrals = {
    'a': {
        'func': lambda x: x / np.sqrt(x**2 + 9),
        'bounds': (0, 4),
        'exact': 2.0,
        'expr': r'$\int_0^4 \frac{x}{\sqrt{x^2+9}} dx$'
    },
    'b': {
        'func': lambda x: (x**3) / (x**2 + 1),
        'bounds': (0, 1),
        'exact': 0.5 - 0.5 * np.log(2),
        'expr': r'$\int_0^1 \frac{x^3}{x^2+1} dx$'
    },
    'c': {
        'func': lambda x: x * np.exp(x),
        'bounds': (0, 1),
        'exact': 1.0,
        'expr': r'$\int_0^1 x e^x dx$'
    },
    'd': {
        'func': lambda x: (x**2) * np.log(x),
        'bounds': (1, 3),
        'exact': 9 * np.log(3) - 26/9,
        'expr': r'$\int_1^3 x^2 \ln x dx$'
    },
    'e': {
        'func': lambda x: (x**2) * np.sin(x),
        'bounds': (0, np.pi),
        'exact': np.pi**2 - 4,
        'expr': r'$\int_0^\pi x^2 \sin x dx$'
    },
    'f': {
        'func': lambda x: (x**3) / np.sqrt(x**4 - 1),
        'bounds': (2, 3),
        'exact': 2 * np.sqrt(5) - 0.5 * np.sqrt(15),
        'expr': r'$\int_2^3 \frac{x^3}{\sqrt{x^4-1}} dx$'
    },
    'g': {
        'func': lambda x: 1.0 / np.sqrt(x**2 + 4),
        'bounds': (0, 2 * np.sqrt(3)),
        'exact': np.log(2 + np.sqrt(3)),
        'expr': r'$\int_0^{2\sqrt{3}} \frac{dx}{\sqrt{x^2+4}}$'
    },
    'h': {
        'func': lambda x: x / np.sqrt(x**4 + 1),
        'bounds': (0, 1),
        'exact': 0.5 * np.log(1 + np.sqrt(2)),
        'expr': r'$\int_0^1 \frac{x \, dx}{\sqrt{x^4+1}}$'
    }
}

# 2. Composite Trapezoid Rule function
def composite_trapezoid(f, a, b, m):
    x = np.linspace(a, b, m + 1)
    y = f(x)
    h = (b - a) / m
    integral = (h / 2) * (y[0] + 2 * np.sum(y[1:-1]) + y[-1])
    return integral

# 3. Compute errors and plot for each integral
fig, axes = plt.subplots(4, 2, figsize=(14, 20))
axes = axes.flatten()

k_values = np.arange(0, 9)
m_values = 2**k_values

print(f"{'Problem':<10}{'Empirical Slope':<18}{'Agrees with Theory (Slope=2)?'}")
print("-" * 55)

for idx, (label, data) in enumerate(integrals.items()):
    a, b = data['bounds']
    f = data['func']
    exact = data['exact']
    
    h_values = (b - a) / m_values
    errors = []
    
    for m in m_values:
        approx = composite_trapezoid(f, a, b, m)
        errors.append(abs(exact - approx))
        
    errors = np.array(errors)
    
    # Calculate the slope ignoring the first two coarse grids for a cleaner asymptotic value
    slope, _ = np.polyfit(np.log(h_values[2:]), np.log(errors[2:]), 1)
    agrees = "Yes" if np.isclose(slope, 2, atol=0.05) else "No"
    print(f"1({label})       {slope:<18.4f}{agrees}")
    
    # Plotting each one log-log style
    ax = axes[idx]
    ax.loglog(h_values, errors, '-o', label='Actual Error', color='darkblue')
    ax.loglog(h_values, h_values**2 * (errors[0]/h_values[0]**2), '--', label='Theoretical $O(h^2)$ Reference', color='crimson')
    
    ax.set_title(f"Integral ({label}): {data['expr']}\nEmpirical Slope = {slope:.4f}", fontsize=11)
    ax.set_xlabel('Step size ($h$)')
    ax.set_ylabel('Absolute Error')
    ax.grid(True, which="both", ls=":")
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
#THIS CELL COMPUTES THE ERRORS OF THE INTEGRALS EVALUATED USING THE COMPOSITE SIMPSON'S RULE

import numpy as np
import matplotlib.pyplot as plt

# 1. Define the integrands (a to h) and their analytical exact values
integrals = {
    'a': {
        'func': lambda x: x / np.sqrt(x**2 + 9),
        'bounds': (0, 4),
        'exact': 2.0,
        'expr': r'$\int_0^4 \frac{x}{\sqrt{x^2+9}} dx$'
    },
    'b': {
        'func': lambda x: (x**3) / (x**2 + 1),
        'bounds': (0, 1),
        'exact': 0.5 - 0.5 * np.log(2),
        'expr': r'$\int_0^1 \frac{x^3}{x^2+1} dx$'
    },
    'c': {
        'func': lambda x: x * np.exp(x),
        'bounds': (0, 1),
        'exact': 1.0,
        'expr': r'$\int_0^1 x e^x dx$'
    },
    'd': {
        'func': lambda x: (x**2) * np.log(x),
        'bounds': (1, 3),
        'exact': 9 * np.log(3) - 26/9,
        'expr': r'$\int_1^3 x^2 \ln x dx$'
    },
    'e': {
        'func': lambda x: (x**2) * np.sin(x),
        'bounds': (0, np.pi),
        'exact': np.pi**2 - 4,
        'expr': r'$\int_0^\pi x^2 \sin x dx$'
    },
    'f': {
        'func': lambda x: (x**3) / np.sqrt(x**4 - 1),
        'bounds': (2, 3),
        'exact': 2 * np.sqrt(5) - 0.5 * np.sqrt(15),
        'expr': r'$\int_2^3 \frac{x^3}{\sqrt{x^4-1}} dx$'
    },
    'g': {
        'func': lambda x: 1.0 / np.sqrt(x**2 + 4),
        'bounds': (0, 2 * np.sqrt(3)),
        'exact': np.log(2 + np.sqrt(3)),
        'expr': r'$\int_0^{2\sqrt{3}} \frac{dx}{\sqrt{x^2+4}}$'
    },
    'h': {
        'func': lambda x: x / np.sqrt(x**4 + 1),
        'bounds': (0, 1),
        'exact': 0.5 * np.log(1 + np.sqrt(2)),
        'expr': r'$\int_0^1 \frac{x \, dx}{\sqrt{x^4+1}}$'
    }
}

# 2. Composite Simpson's Rule function
def composite_simpson(f, a, b, m):
    # m must be even for Simpson's rule
    if m % 2 != 0:
        raise ValueError("Number of panels m must be even.")
    x = np.linspace(a, b, m + 1)
    y = f(x)
    h = (b - a) / m
    integral = (h / 3) * (y[0] + 4 * np.sum(y[1:-1:2]) + 2 * np.sum(y[2:-1:2]) + y[-1])
    return integral

# 3. Compute errors and plot for each integral
fig, axes = plt.subplots(4, 2, figsize=(14, 20))
axes = axes.flatten()

# For Simpson's rule, start from k=1 (m=2) to guarantee even panel counts
k_values = np.arange(1, 9)
m_values = 2**k_values

print(f"{'Problem':<10}{'Empirical Slope':<18}{'Agrees with Theory (Slope=4)?'}")
print("-" * 55)

for idx, (label, data) in enumerate(integrals.items()):
    a, b = data['bounds']
    f = data['func']
    exact = data['exact']
    
    h_values = (b - a) / m_values
    errors = []
    
    for m in m_values:
        approx = composite_simpson(f, a, b, m)
        error = abs(exact - approx)
        # Avoid log(0) if it hits machine precision exactly
        errors.append(max(error, 1e-16))
        
    errors = np.array(errors)
    
    # Track the active asymptotic region (ignore flatlines due to machine precision limits)
    valid_idx = errors > 1e-14
    if np.sum(valid_idx) > 2:
        slope, _ = np.polyfit(np.log(h_values[valid_idx]), np.log(errors[valid_idx]), 1)
    else:
        slope, _ = np.polyfit(np.log(h_values[:4]), np.log(errors[:4]), 1)
        
    agrees = "Yes" if np.isclose(slope, 4, atol=0.15) else "No"
    print(f"1({label})       {slope:<18.4f}{agrees}")
    
    # Plotting each one log-log style
    ax = axes[idx]
    ax.loglog(h_values, errors, '-o', label='Simpson Error', color='darkgreen')
    ax.loglog(h_values, h_values**4 * (errors[0]/h_values[0]**4), '--', label='Theoretical $O(h^4)$ Reference', color='crimson')
    
    ax.set_title(f"Integral 1({label}): {data['expr']}\nEmpirical Slope = {slope:.4f}", fontsize=11)
    ax.set_xlabel('Step size ($h$)')
    ax.set_ylabel('Absolute Error')
    ax.grid(True, which="both", ls=":")
    ax.legend()

plt.tight_layout()
plt.show()
